In [58]:
import pandas as pd

In [59]:
DATA = '../data/'

In [60]:
elo_featured = pd.read_parquet(DATA+'elo_featured.parquet')
display(elo_featured.head())

,global_rank,rating,rank_max,rating_max,rank_avg,rating_avg,rank_min,rating_min,rank_3m_change,rating_3m_change,...,rank_change,rating_change,team_name,year_applied,matches_total_year,wins_year,losses_year,draws_year,goals_for_year,goals_against_year
4372,110,1302,110,1302,110,1302,110,1302,0.0,0.0,...,0.0,0.0,Aden,1962,1.0,1.0,0.0,0.0,4.0,2.0
4527,110,1302,110,1302,111,1302,111,1302,0.0,0.0,...,0.0,0.0,Aden,1963,0.0,0.0,0.0,0.0,0.0,0.0
4684,111,1302,109,1302,110,1302,111,1302,0.0,0.0,...,-1.0,0.0,Aden,1964,0.0,0.0,0.0,0.0,0.0,0.0
4845,113,1302,109,1302,111,1302,114,1302,1.0,0.0,...,-2.0,0.0,Aden,1965,0.0,0.0,0.0,0.0,0.0,0.0
5018,126,1249,109,1302,113,1298,128,1249,-1.0,0.0,...,-13.0,-53.0,Aden,1966,4.0,0.0,4.0,0.0,3.0,25.0


In [61]:
train = pd.read_parquet(DATA+'train.parquet')
train.head()

,Year,Date,Home Team,Home Goals,Away Goals,Away Team,home_winner
0,1930,1930-07-13,France,4,1,Mexico,1
1,1930,1930-07-13,United States,3,0,Belgium,1
2,1930,1930-07-14,Yugoslavia,2,1,Brazil,1
3,1930,1930-07-14,Romania,3,1,Peru,1
4,1930,1930-07-15,Argentina,1,0,France,1


In [62]:
def enrich_data(df, enrich):
    _df = df.copy()

    # enrich home team
    _df = _df.merge(
        enrich.add_prefix('home_').rename(columns={'home_team_name' : 'Home Team', 'home_year_applied' : 'Year'}),
        on=['Home Team', 'Year'],
        how='left'
    )

    # enrich away team
    _df = _df.merge(
        enrich.add_prefix('away_').rename(columns={'away_team_name' : 'Away Team', 'away_year_applied' : 'Year'}),
        on=['Away Team', 'Year'],
        how='left'
    )

    _df.columns = _df.columns.str.lower().str.replace(' ', '_')
    return _df

In [63]:
country_name_maps = {
    'Czech Republic' : 'Czechia',
    'DR Congo' : 'Congo',
    'Republic of Ireland' : 'Ireland',
}

train['Home Team'] = (
    train['Home Team']
    .map(country_name_maps)
    .fillna(train['Home Team'])
)
train['Away Team'] = (
    train['Away Team']
    .map(country_name_maps)
    .fillna(train['Away Team'])
)

In [ ]:
train_enriched = enrich_data(train, elo_featured)
train_enriched.fillna(0) # alguns anos estão sem elo por motivos de guerra ou similares

train_enriched.head()

,year,date,home_team,home_goals,away_goals,away_team,home_winner,home_global_rank,home_rating,home_rank_max,...,away_goals_for,away_goals_against,away_rank_change,away_rating_change,away_matches_total_year,away_wins_year,away_losses_year,away_draws_year,away_goals_for_year,away_goals_against_year
0,1930,1930-07-13,France,4,1,Mexico,1,37.0,1547.0,9.0,...,17.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1930,1930-07-13,United States,3,0,Belgium,1,21.0,1672.0,13.0,...,238.0,287.0,0.0,-4.0,5.0,2.0,3.0,0.0,8.0,12.0
2,1930,1930-07-14,Yugoslavia,2,1,Brazil,1,28.0,1608.0,25.0,...,63.0,58.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1930,1930-07-14,Romania,3,1,Peru,1,33.0,1572.0,23.0,...,5.0,23.0,-7.0,-37.0,3.0,0.0,3.0,0.0,1.0,12.0
4,1930,1930-07-15,Argentina,1,0,France,1,1.0,2081.0,1.0,...,156.0,341.0,3.0,2.0,6.0,2.0,4.0,0.0,9.0,19.0
